# Session 7 · Spindles and slow waves

That's the point of this session: running a detector takes one line. **What you count depends on the parameters**. You will:
1. detect spindles in one night with **YASA** (https://yasa-sleep.org) and look at what it finds
2. **turn the knobs** of the spindle detector: thresholds, frequency band, channel; and measure its false alarms
3. detect slow waves, and see what a **fixed amplitude threshold** does to a young and an old brain
4. measure the **coupling** between slow waves and spindles

**How to work**
- Mostly **run, change, look**: cells marked 👉 have a parameter to change and rerun. The `# TODO` cells are short.
- Nights: **SC4002** (33 y) and **SC4752** (96 y), as on Tuesday.
- Course rule (Session 3): events are only detected in the **clean** epochs; the bad ones are left out.

In [ ]:
%matplotlib inline
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa

sys.path.insert(0, str(Path("..").resolve()))  # so Python finds sleep_utils.py and cohort.py in the repo folder
from sleep_utils import SUBJECTS_CSV, YASA_CODES, load_continuous

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

subjects = pd.read_csv(SUBJECTS_CSV)
SPINDLE_BAND = (12, 16)  # Hz, our spindle band (YASA's default is 12-15 Hz)

YASA prints a `WARNING` when it finds no event in a channel (e.g. *No SW were found in channel Pz-Oz* for an old person). That's information about the night, not an error.

## 1. Spindles in one night

YASA's detectors need the **continuous** signal (not 30-s epochs: a spindle can cross an epoch border) and a hypnogram with **one value per sample**. `load_continuous` gives both. It is `load_epochs` (load, filter 0.1–40 Hz, bad epochs) plus that hypnogram, with YASA's integer codes (0 WAKE, 1 N1, 2 N2, 3 N3, 4 REM) and **−1 in the bad epochs**, so that YASA skips them.

**How YASA finds a spindle** (Vallat & Walker, 2021; details in T8). In sliding windows, three measures of the signal:
- `rel_pow`: the **relative power** in the spindle band (`freq_sp`) out of 1–30 Hz (2-s windows). Default threshold 0.2.
- `corr`: the **correlation** between the signal filtered in the spindle band and the broadband signal (0.3-s windows): is the spindle visible in the raw EEG? Default 0.65.
- `rms`: the **root mean square** of the spindle-band signal (0.3-s windows), above its mean + 1.5 standard deviations (over the N2/N3 of the night). Default 1.5.

**Spindle band:** we use **12–16 Hz** (`SPINDLE_BAND`, set in the first cell), the usual band for human spindles; YASA's default is 12–15 Hz. All the other parameters are YASA's defaults.

Where all three thresholds are crossed, an event of **0.5–2 s** (`duration`) is a spindle. Spindles closer than 500 ms (`min_distance`) are merged.

In [ ]:
SUBJECT, NIGHT = 0, 2  # SC4002, 33 years

raw, hypno, stages, artefacts = load_continuous(SUBJECT, NIGHT)  # raw: Fpz-Cz and Pz-Oz only, filtered 0.1-40 Hz
print(f"{len(stages)} epochs, {artefacts['bad'].sum()} bad; channels {raw.ch_names}")
print("clean minutes:", {s: (hypno == c).sum() / raw.info["sfreq"] / 60 for s, c in YASA_CODES.items() if c >= 0})

sp = yasa.spindles_detect(raw, hypno=hypno, include=(2, 3), freq_sp=SPINDLE_BAND)  # in the clean N2 and N3 epochs
spindles = sp.summary()  # one row per spindle
spindles.head()

One row per spindle: when (`Start`, `Peak`, `End`, in s from lights-off), how long, how big (`Amplitude`: peak-to-peak in µV), how fast (`Frequency`), how many cycles (`Oscillations`), which `Stage` and `Channel`. Per stage and channel, with the **density** (spindles per minute of that stage):

In [ ]:
sp.summary(grp_chan=True, grp_stage=True).round(2)

### Look at them

The detected spindles in red. 👉 Change `K` (which spindle to centre on), `CHANNEL`, or the window length.

In [ ]:
def plot_events(raw, results, t0, duration=20, channel="Fpz-Cz", title=""):
    """`duration` s of `channel` from `t0` (s since lights-off), with the events YASA detected in red."""
    sf = raw.info["sfreq"]
    i0, i1 = int(t0 * sf), int((t0 + duration) * sf)
    x = raw.get_data(picks=[channel], units="uV")[0, i0:i1]
    mask = results.get_mask()[raw.ch_names.index(channel), i0:i1].astype(bool)  # 1 inside an event
    t = np.arange(i0, i1) / sf
    fig, ax = plt.subplots(figsize=(14, 2.8))
    ax.plot(t, x, color="black", lw=0.6)
    ax.plot(t, np.where(mask, x, np.nan), color="red", lw=1)
    ax.set(xlabel="s since lights-off", ylabel=f"{channel} (µV)", title=title)
    plt.show()


K = 100  # 👉 the K-th spindle
CHANNEL = "Fpz-Cz"  # 👉 or "Pz-Oz"
on_channel = spindles[spindles["Channel"] == CHANNEL].reset_index(drop=True)
plot_events(raw, sp, on_channel.loc[K, "Peak"] - 10, channel=CHANNEL,
            title=f"Spindle {K}: {on_channel.loc[K, 'Frequency']:.1f} Hz, {on_channel.loc[K, 'Amplitude']:.0f} µV")

And all of them **averaged**, aligned on their largest peak:

In [ ]:
sp.plot_average(center="Peak", time_before=1, time_after=1, hue="Channel")
plt.show()

**Questions**
1. Look at 5–10 spindles (👉 `K`). Would you have marked them? Did you see spindles in the window that YASA missed?
2. How many spindles per minute of N2, per channel? Why so different between Fpz-Cz and Pz-Oz?
3. Why are there spindles in N3 at all, and why so few?

## 2. Turn the knobs: spindles

From now on, both nights side by side. This small function runs a detector and returns, per channel, the **density** (events per minute of the included clean stages), the mean **amplitude** and **frequency**:

In [ ]:
NIGHTS = {"SC4002 (33 y)": (0, 2), "SC4752 (96 y)": (75, 2)}
data = {label: load_continuous(s, n) for label, (s, n) in NIGHTS.items()}  # label -> (raw, hypno, stages, artefacts)


def detect(kind, raw, hypno, include=(2,), **params):
    """YASA's spindles (kind="sp") or slow waves ("sw") as a table of events (empty table if there are none)."""
    if kind == "sp":
        params = {"freq_sp": SPINDLE_BAND, **params}  # our band, unless params gives another one
    detector = yasa.spindles_detect if kind == "sp" else yasa.sw_detect
    results = detector(raw, hypno=hypno, include=include, **params)
    return results.summary() if results is not None else pd.DataFrame(columns=["Stage", "Channel"])


def event_stats(kind, raw, hypno, include=(2,), **params):
    """Per channel: density (per minute of clean `include` stages), mean amplitude and frequency, as a dict."""
    events = detect(kind, raw, hypno, include, **params)
    minutes = np.isin(hypno, include).sum() / raw.info["sfreq"] / 60
    amplitude = "Amplitude" if kind == "sp" else "PTP"  # spindles: peak-to-peak; slow waves: negative to positive peak
    stats = {}
    for ch in raw.ch_names:
        ev = events[events["Channel"] == ch]
        stats[f"{ch} density"] = len(ev) / minutes
        stats[f"{ch} amplitude"] = ev[amplitude].mean() if len(ev) else np.nan
        stats[f"{ch} frequency"] = ev["Frequency"].mean() if len(ev) else np.nan
    return stats

### Exercise 2.1: settings

👉 Run the table, then add your own rows to `SP_CONFIGS`. Any argument of `yasa.spindles_detect` works: `thresh=dict(rel_pow=..., corr=..., rms=...)` (the ones you don't give keep their default), `freq_sp=(low, high)`, `duration=(min, max)`, `min_distance` (ms).

In [ ]:
SP_CONFIGS = {  # 👉 add your own
    "course settings": dict(),  # 12-16 Hz, YASA's default thresholds
    "rel_pow 0.3": dict(thresh=dict(rel_pow=0.3)),
    "corr 0.8": dict(thresh=dict(corr=0.8)),
    "rms 3": dict(thresh=dict(rms=3)),
    "YASA's band 12-15 Hz": dict(freq_sp=(12, 15)),
    "duration 0.3-3 s": dict(duration=(0.3, 3)),
}

rows = {}
for config, params in SP_CONFIGS.items():
    for label, (raw_, hypno_, _, _) in data.items():
        rows[(config, label)] = event_stats("sp", raw_, hypno_, include=(2,), **params)
sp_table = pd.DataFrame(rows).T
sp_table.round(2)

**Questions**
1. Which parameter changes the density most? Which one hardly matters?
2. With the course settings, does the 96-year-old have fewer spindles than the 33-year-old? Smaller ones? Faster or slower? Is that what you expect?
3. How much does the band (12–16 vs. YASA's 12–15 Hz) change? For which person, and why?

### Exercise 2.2: sensitivity curve

One parameter, many values, both nights. 👉 Change `PARAM`, `VALUES`, `CHANNEL`.

In [ ]:
PARAM = "corr"  # 👉 "rel_pow" (try 0.1-0.4), "corr" (0.5-0.9), "rms" (1-3)
VALUES = [0.5, 0.6, 0.65, 0.7, 0.8, 0.9]  # 👉
CHANNEL = "Fpz-Cz"  # 👉

curves = {}
for label, (raw_, hypno_, _, _) in data.items():
    curves[label] = [event_stats("sp", raw_, hypno_, include=(2,), thresh={PARAM: v})[f"{CHANNEL} density"]
                     for v in VALUES]
curves = pd.DataFrame(curves, index=pd.Index(VALUES, name=PARAM))
curves["ratio young / old"] = curves.iloc[:, 0] / curves.iloc[:, 1]

ax = curves.iloc[:, :2].plot(marker="o", figsize=(7, 4), ylabel="spindles per min of N2",
                             title=f"Spindle density vs. {PARAM}, {CHANNEL}")
plt.show()
curves.round(2)

**Questions**
1. If you had to pick the `corr` threshold for a study, which value would you choose? Can these curves tell you which one is right? What would you need instead?
2. Look at the ratio column. Is the difference between the two people the same at every threshold? Repeat with `rel_pow`, and on Pz-Oz. When can you trust a difference between two people (or two groups) measured with a detector?
3. A paper reports a spindle density of 2.5 per min in young adults; you find 1.3 in SC4002. What do you need to know before you compare the two numbers?

### Exercise 2.3: false alarms

A detector also needs a **specificity** check: run it where there should be no spindles. Classic spindles are absent in WAKE and REM (and rare in N1). Everything found there is a false alarm, or something else (alpha, eye movements, muscle).

The cell shows two tables, one row per stage and one column per channel:
- `density`: events per minute of clean signal in that stage (in N2 it's the usual spindle density; elsewhere it's the false-alarm rate)
- `frequency`: the mean frequency of those events, to see whether the "spindles" outside N2/N3 look like the N2 ones or like something else (e.g. alpha, 8–12 Hz)

In [ ]:
label = "SC4752 (96 y)"  # 👉 or "SC4002 (33 y)"
raw_, hypno_, _, _ = data[label]
events = detect("sp", raw_, hypno_, include=(0, 1, 2, 3, 4))  # all stages
counts = events.groupby(["Stage", "Channel"]).size().unstack(fill_value=0)

# TODO: the clean minutes of each stage (codes 0-4), as a Series indexed by the stage code. Hint: (hypno_ == code).sum()
minutes = ...
density = counts.div(minutes, axis=0).rename(index={c: s for s, c in YASA_CODES.items()})
frequency = events.groupby(["Stage", "Channel"])["Frequency"].mean().unstack().rename(index={c: s for s, c in YASA_CODES.items()})
print(label, "- events per minute, by stage")
display(density.round(2))
print(label, "- mean frequency of the events (Hz), by stage")
display(frequency.round(1))

In [ ]:
# 👉 Optional: plot the two tables (e.g. a bar chart of density by stage, one bar per channel)

**Questions**
1. In which night does the detector find "spindles" in REM or WAKE? On which channel? What could they be? (Look at one with `plot_events`.)
2. If the old people's N2 contains some of the same false alarms, what does it do to an age comparison of spindle density?

### Exercise 2.4: which threshold does the work?

A spindle needs **all three** criteria at the same time (in YASA 0.8.1):
- `rel_pow`: the sigma band's share of the 1–30 Hz power (2-s windows, every 200 ms) ≥ 0.2
- `corr`: the correlation between the sigma-filtered and the broadband signal (0.3-s windows) ≥ 0.65: the signal *looks like* its sigma component
- `rms`: the moving RMS of the sigma-filtered signal (0.3-s windows) ≥ mean + `rms` × SD of that RMS over the included stage (a trimmed SD, without the 10 % extremes), **capped at 10 µV**

The first two are about the **shape** of the signal (they don't change if you double its amplitude); the third is about **amplitude**, measured against the night's own sigma activity. In 2.1, `rms` 1.5 → 3 hardly changed anything. Why?

👉 Run the table: `None` switches a criterion off. Add your own rows.

In [ ]:
KNOCKOUT = {  # 👉 thresholds; None = criterion switched off
    "all three (default)": dict(),
    "without rms": dict(rms=None),
    "without corr": dict(corr=None),
    "without rel_pow": dict(rel_pow=None),
    "rms only": dict(rel_pow=None, corr=None),
    "rms 5": dict(rms=5),
    "rms 100": dict(rms=100),
}

rows = {}
for config, thresh in KNOCKOUT.items():
    for label, (raw_, hypno_, _, _) in data.items():
        rows[(config, label)] = event_stats("sp", raw_, hypno_, include=(2,), thresh=thresh)
pd.DataFrame(rows).T.round(2)

👉 Now look inside the detector: the three criteria, computed as YASA does, over 20 s of N2 around one spindle. Dashed lines = thresholds, red bands = detected spindles. Change the night, the channel and the spindle.

In [ ]:
from yasa import moving_transform, stft_power
from yasa.others import trimbothstd

label = "SC4002 (33 y)"  # 👉 or "SC4752 (96 y)"
CHANNEL = "Fpz-Cz"  # 👉 or "Pz-Oz"
SPINDLE = 30  # 👉 the window is centred on this spindle (0, 1, 2, ...)
THRESH = dict(rel_pow=0.2, corr=0.65, rms=1.5)  # YASA's defaults

raw_, hypno_, _, _ = data[label]
sf = raw_.info["sfreq"]
x = raw_.get_data(picks=[CHANNEL], units="uV")[0]

# The three criteria, as in yasa.spindles_detect
broad = mne.filter.filter_data(x, sf, 1, 30, verbose=False)
sigma = mne.filter.filter_data(x, sf, *SPINDLE_BAND, l_trans_bandwidth=1.5, h_trans_bandwidth=1.5, verbose=False)
f, t, Sxx = stft_power(broad, sf, window=2, step=0.2, band=(1, 30), interp=False, norm=False)
in_band = (f >= SPINDLE_BAND[0]) & (f <= SPINDLE_BAND[1])
rel_pow = np.interp(np.arange(len(x)) / sf, t, Sxx[in_band].sum(0) / Sxx.sum(0))
_, corr = moving_transform(sigma, broad, sf, window=0.3, step=0.1, method="corr", interp=True)
_, rms = moving_transform(sigma, sf=sf, window=0.3, step=0.1, method="rms", interp=True)

# The RMS threshold in µV: relative to this night's N2, capped at 10 µV
n2 = hypno_ == 2
mean, sd = rms[n2].mean(), trimbothstd(rms[n2], cut=0.1)
print(f"{label}, {CHANNEL}: moving sigma RMS in N2 = {mean:.2f} ± {sd:.2f} µV")
for k in [1.5, 3, 5]:
    print(f"  rms={k}: threshold {min(mean + k * sd, 10):.2f} µV")
print(f"  the 10 µV cap is reached at rms={(10 - mean) / sd:.1f}")
rms_thresh = min(mean + THRESH["rms"] * sd, 10)

# Plot 20 s around one spindle
spindles = detect("sp", raw_.copy().pick([CHANNEL]), hypno_, include=(2,), thresh=THRESH)
centre = spindles["Peak"].iloc[SPINDLE]
win = slice(int((centre - 10) * sf), int((centre + 10) * sf))
time = np.arange(len(x))[win] / sf

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
axes[0].plot(time, broad[win], color="k", lw=0.6, label="1-30 Hz")
axes[0].plot(time, sigma[win], color="tab:blue", lw=0.8, label=f"{SPINDLE_BAND[0]}-{SPINDLE_BAND[1]} Hz")
axes[0].set_title(f"{label}, {CHANNEL}: EEG (µV)", fontsize=10, loc="left")
axes[0].legend(fontsize=8, loc="upper right")
criteria = [(rel_pow, THRESH["rel_pow"], "rel_pow: sigma / total power"),
            (corr, THRESH["corr"], "corr: sigma vs. broadband"),
            (rms, rms_thresh, f"rms: moving sigma RMS (µV), threshold {rms_thresh:.1f} µV")]
for ax, (trace, threshold, name) in zip(axes[1:], criteria):
    ax.plot(time, trace[win], color="k", lw=0.8)
    ax.axhline(threshold, color="tab:red", ls="--")
    ax.set_title(name, fontsize=10, loc="left")
for _, sp in spindles[(spindles["End"] > time[0]) & (spindles["Start"] < time[-1])].iterrows():
    for ax in axes:
        ax.axvspan(sp["Start"], sp["End"], color="tab:red", alpha=0.15)
axes[-1].set_xlabel("time since lights-off (s)")
axes[-1].set_xlim(time[0], time[-1])
plt.tight_layout()
plt.show()

**Questions**
1. Which criterion can you switch off without changing much? Which one alone finds the most "spindles"? Look at the amplitude of the extra events too.
2. In the plot, move through a few spindles (and both people). Where `rel_pow` and `corr` are both above their line, where is the RMS? So why does `rms` 1.5 → 3 hardly change anything?
3. What does `rms=1.5` mean in µV for each person and channel? Is the RMS criterion an amplitude criterion that is the same for everybody? And with `rms=100`?
4. So is `rms` useless? When could it matter?

## 3. Slow waves

**How YASA finds a slow wave** (after Massimini et al., 2004; Carrier et al., 2011): the signal is filtered to **0.3–1.5 Hz** (`freq_sw`); every negative peak followed by a positive peak is a candidate. It is a slow wave if
- the **negative peak** is between −40 and −200 µV (`amp_neg`), the **positive** peak between 10 and 150 µV (`amp_pos`)
- the **peak-to-peak** amplitude is between 75 and 350 µV (`amp_ptp`)
- the negative half-wave lasts 0.3–1.5 s (`dur_neg`), the positive one 0.1–1 s (`dur_pos`).

The 75 µV comes from the R&K and AASM definition of a slow wave for N3 scoring (in AASM: measured over the frontal regions). Note the units: these thresholds are **µV**, not relative to the person's own signal.

In [ ]:
sw_rows = {label: event_stats("sw", raw_, hypno_, include=(2, 3)) for label, (raw_, hypno_, _, _) in data.items()}
print("Slow waves in the clean N2 + N3 epochs (defaults)")
display(pd.DataFrame(sw_rows).T.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
for label, (raw_, hypno_, _, _) in data.items():
    sw_ = yasa.sw_detect(raw_.copy().pick(["Fpz-Cz"]), hypno=hypno_, include=(2, 3))
    average = sw_.get_sync_events(center="NegPeak", time_before=1, time_after=1.5).groupby("Time")["Amplitude"].mean()
    ax.plot(average.index, average, label=f"{label}: {len(sw_.summary())} slow waves")
ax.axhline(0, color="gray", lw=0.5)
ax.set(xlabel="s from the negative peak", ylabel="Fpz-Cz (µV)", title="Average slow wave (default detection)")
ax.legend()
plt.show()

**Questions**
1. How many slow waves per minute of N2+N3 in each night? Where do the old person's few slow waves come from (stage, channel)?
2. Compare the average slow waves: amplitude, duration, slope. What should be different with age, and what can't be (look at the thresholds)?

### Exercise 3.1: fewer slow waves, or smaller ones?

The density for a range of amplitude thresholds. To go below 75 µV peak-to-peak, the negative-peak limit must come down too (`amp_neg` starts at 40 µV by default), so we set it to 10 µV for all thresholds. 👉 Change `THRESHOLDS`, `STAGE` (3 = N3, 2 = N2) or `CHANNEL`.

In [ ]:
THRESHOLDS = [20, 30, 40, 50, 60, 75, 100, 125, 150]  # 👉 minimum peak-to-peak amplitude, µV
STAGE = 3  # 👉 3 = N3, 2 = N2
CHANNEL = "Fpz-Cz"  # 👉

curves = {}
for label, (raw_, hypno_, _, _) in data.items():
    curves[label] = [event_stats("sw", raw_, hypno_, include=(STAGE,), amp_neg=(10, 200), amp_ptp=(t, 350))[
        f"{CHANNEL} density"] for t in THRESHOLDS]
curves = pd.DataFrame(curves, index=THRESHOLDS)
curves["ratio young / old"] = curves.iloc[:, 0] / curves.iloc[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
curves.iloc[:, :2].plot(ax=axes[0], marker="o", xlabel="minimum peak-to-peak amplitude (µV)",
                        ylabel="slow waves per min", title=f"Slow-wave density, stage {STAGE}, {CHANNEL}")
axes[0].axvline(75, color="gray", ls="--")
for label, (raw_, hypno_, _, _) in data.items():  # the amplitude of every candidate wave
    every_wave = detect("sw", raw_.copy().pick([CHANNEL]), hypno_, include=(STAGE,), amp_neg=(10, 200), amp_ptp=(10, 350))
    axes[1].hist(every_wave["PTP"], bins=np.arange(10, 300, 5), density=True, histtype="step", lw=1.5, label=label)
axes[1].axvline(75, color="gray", ls="--")
axes[1].set(xlabel="peak-to-peak amplitude (µV)", ylabel="fraction of waves", title="All candidate waves")
axes[1].legend()
plt.tight_layout()
plt.show()
curves.round(2)

**Questions**
1. At the default 75 µV, how many times more slow waves per minute of N3 does the young person have? And at 20 µV?
2. Look at the histogram: are the old person's waves rarer, or smaller?
3. "Slow-wave density falls with age": is that a statement about the brain, or about the threshold? What else, besides the brain, makes an EEG smaller in an old person?
4. 🐭 In rodent EEG/LFP, amplitudes depend on the electrode (screw vs. depth, impedance) and the animal. Would you use a fixed µV threshold there?

### Exercise 3.2: other settings

👉 Add rows to `SW_CONFIGS`: e.g. `freq_sw=(0.5, 2)` ("delta waves") vs. `(0.3, 1)` ("slow oscillations"), `amp_ptp=(40, 350)`, `dur_neg=(0.25, 1.0)`.

In [ ]:
SW_CONFIGS = {  # 👉 add your own
    "defaults": dict(),
    "slow oscillations 0.3-1 Hz": dict(freq_sw=(0.3, 1)),
    "delta waves 0.5-2 Hz": dict(freq_sw=(0.5, 2)),
    "PTP >= 40 µV": dict(amp_neg=(20, 200), amp_ptp=(40, 350)),
}

rows = {}
for config, params in SW_CONFIGS.items():
    for label, (raw_, hypno_, _, _) in data.items():
        rows[(config, label)] = event_stats("sw", raw_, hypno_, include=(2, 3), **params)
pd.DataFrame(rows).T.round(2)

**Question.** Slow oscillations (< 1 Hz) and delta waves (1–4 Hz) are thought to be partly different phenomena (Kim et al., 2019: slow oscillations help consolidate memories, delta waves help forget). Which of the two does the old person keep?

## 4. Slow waves and spindles together: coupling

Memory consolidation is thought to need spindles at the right moment of the slow wave: around the **up-state** (the positive peak), when the cortex is depolarised. YASA measures this with `coupling=True`, for every slow wave:
- `PhaseAtSigmaPeak`: the **phase** of the slow wave (0.3–1.5 Hz) at the moment the spindle-band (12–16 Hz) amplitude is largest, within ±1 s of the negative peak. **0° = the up-state peak**, ±180° = the down-state (negative peak); negative phases are on the way up, positive ones on the way down.
- `ndPAC`: the strength of phase–amplitude coupling within that window (normalised direct PAC).

Phases are **circular** (179° and −179° are neighbours): their mean is the direction of the mean vector, and its length (0–1) says how consistent they are.

In [ ]:
def circular_mean(phases):
    """Mean direction (radians) and its strength (0 = phases all over the place, 1 = all the same)."""
    vector = np.mean(np.exp(1j * np.asarray(phases)))  # each phase as a unit vector; their mean
    return np.angle(vector), np.abs(vector)


CHANNEL = "Fpz-Cz"  # 👉
COUPLING = dict(freq_sp=(12, 16), time=1)  # 👉 YASA's defaults: spindle band, ± seconds around the negative peak

fig, axes = plt.subplots(1, 2, figsize=(11, 5), subplot_kw=dict(projection="polar"))
for ax, (label, (raw_, hypno_, _, _)) in zip(axes, data.items()):
    events = detect("sw", raw_.copy().pick([CHANNEL]), hypno_, include=(2, 3), coupling=True, coupling_params=COUPLING)
    direction, strength = circular_mean(events["PhaseAtSigmaPeak"])
    ax.hist(events["PhaseAtSigmaPeak"], bins=36, density=True, color="lightgray", edgecolor="gray")
    ax.annotate("", xy=(direction, strength * ax.get_ylim()[1] * 3), xytext=(0, 0),
                arrowprops=dict(color="red", width=2))
    ax.set_title(f"{label}\n{len(events)} slow waves: {np.degrees(direction):.0f}°, strength {strength:.2f}, "
                 f"ndPAC {events['ndPAC'].mean():.2f}", fontsize=10)
plt.tight_layout()
plt.show()

**Questions**
1. Where in the slow wave do the young person's spindles peak? And the old person's? Is it the direction reported in the literature?
2. How much would you trust the old person's numbers? Change the amplitude threshold of the slow waves (add e.g. `amp_neg=(10, 200), amp_ptp=(30, 350)` to `detect`): does the phase stay?
3. The phase is computed for **every** slow wave, also when there's no spindle anywhere near. What does that do to the strength?

## Optional exercises

1. **Artefacts.** Detect spindles and slow waves in SC4002 **without** removing the bad epochs (rebuild `hypno` from `stages` with `YASA_CODES`, no −1 for the detector's bad epochs). How much do the densities change? Why so little? (Hint: what limits do YASA's detectors have on their own?) Try a night with more bad N2/N3 epochs, e.g. SC4651 or SC4191.
2. **Where are the spindles in the night?** Spindle density per sleep cycle, or per hour since sleep onset (`spindles["Start"]` in s from lights-off). Does it change across the night? Same for slow-wave density and amplitude: the homeostatic decline of slow-wave activity (Process S) should show.
3. **Slow and fast spindles.** Split the band into slow (11–13 Hz) and fast (13–16 Hz) spindles (add two rows to `SP_CONFIGS` with `freq_sp`). Which kind does each person have, and which channel sees which? Slow spindles are frontal, fast ones centro-parietal; with our two bipolar channels (Fpz-Cz, Pz-Oz) this topography is hard to see: why?
4. **An exceptional sleeper.** SC4191 (28 y) has ~4 times more spindles per minute than SC4002. Look at their spindles with `plot_events`, and at their N2 spectrum: is it real?

In [ ]:
# Optional exercise 1

In [ ]:
# Optional exercise 2

In [ ]:
# Optional exercise 3

In [ ]:
# Optional exercise 4

## What's next

Next: the **cohort pipeline**, which brings the features of the week into one table (one row per night), and then machine learning on that table.